# Clase 13 — Test A/B y diseño de experimentos

**Curso:** Estadística Aplicada con Python · **Bloque 4:** Modelos y experimentos
**Tiempo estimado:** 105 minutos · **Capítulo del manual:** `manual/13_ab_testing_diseno_experimentos.md`

## Qué vas a ver en esta clase

- Por qué la **aleatorización** permite hablar de causa (en promedio) y qué pasa cuando la gente se autoselecciona.
- **Diseñar** el experimento antes de empezar: efecto mínimo relevante, α, potencia y **tamaño de muestra**, a mano y con `statsmodels`.
- **Analizar** un test A/B de conversión: test de dos proporciones, intervalo de confianza de la diferencia y *lift*.
- El ***peeking***: mirar los resultados antes de tiempo y parar al primer p < 0,05.
- **Controles de salud**: SRM (sample ratio mismatch) y test A/A.
- La **paradoja de Simpson** y los **segmentos y métricas múltiples**.

## Requisitos

- Clase 7 (test de dos proporciones, errores I y II, potencia como idea), clase 9 (chi-cuadrado y comparaciones múltiples), clase 6 (intervalos de confianza).
- Librerías: `numpy`, `pandas`, `matplotlib`, `scipy`, `statsmodels` (`pip install numpy pandas matplotlib scipy statsmodels`; ya vienen instaladas en Google Colab).

> **Cómo usar este notebook:** ejecuta las celdas en orden. Antes de cada experimento escribe qué esperas que ocurra. Todos los datos son sintéticos (*ejemplo ilustrativo*) y sabemos el efecto verdadero, para poder comprobar cuándo el método acierta y cuándo no.

---
## Celda 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_effectsize, confint_proportions_2indep
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings("ignore")

rng = np.random.default_rng(42)          # semilla fija: todos obtenemos los mismos resultados
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

def n_por_grupo(p1, p2, alfa=0.05, potencia=0.80):
    """Tamaño de muestra por grupo para comparar dos proporciones (fórmula normal, sin redondear)."""
    za, zb = stats.norm.ppf(1 - alfa / 2), stats.norm.ppf(potencia)
    return (za + zb) ** 2 * (p1 * (1 - p1) + p2 * (1 - p2)) / (p2 - p1) ** 2

def z_controles(rng, nsim, controles, n_final=3840, p=0.10):
    """Simula nsim experimentos A/A (sin efecto): estadístico z acumulado en cada uno de los controles."""
    por = n_final // controles
    xa = rng.binomial(por, p, (nsim, controles)).cumsum(axis=1)
    xb = rng.binomial(por, p, (nsim, controles)).cumsum(axis=1)
    ns = por * np.arange(1, controles + 1)
    pa, pb, pp = xa / ns, xb / ns, (xa + xb) / (2 * ns)
    return (pb - pa) / np.sqrt(pp * (1 - pp) * 2 / ns)

def falsos_positivos(rng, nsim, controles, umbral=1.96):
    """Porcentaje de experimentos A/A declarados 'significativos' en algún control."""
    sig = np.abs(z_controles(rng, nsim, controles)) > umbral
    return sig.any(axis=1).mean() * 100

def z_dos_proporciones(xa, na, xb, nb):
    """Estadístico z de dos proporciones con varianza combinada (vectorizado)."""
    pa, pb, pp = xa / na, xb / nb, (xa + xb) / (na + nb)
    return (pb - pa) / np.sqrt(pp * (1 - pp) * (1 / na + 1 / nb))

print("Librerías y funciones cargadas correctamente.")

---
## 1. Por qué aleatorizar

Una tienda en línea quiere saber si un **nuevo botón de pago** (tratamiento) sube la conversión. Fabricamos una población de 200 000 visitantes donde conocemos, para cada uno, qué haría **con** y **sin** el botón (algo que en la vida real nunca vemos). Así el efecto verdadero es un número conocido y podemos juzgar dos formas de repartir el botón. Además cada visitante tiene un **interés previo** en comprar, que la empresa no observa y que aumenta la probabilidad de comprar.

### Experimento guiado 1: asignación aleatoria frente a autoselección

**Antes de ejecutar, predice:** si dejamos que los visitantes más interesados usen el botón nuevo (autoselección), ¿el efecto estimado será mayor, menor o igual que el verdadero? ¿Y si lo asignamos con una moneda?

In [ ]:
rng_pob = np.random.default_rng(42)
M = 200_000
interes = rng_pob.normal(size=M)                       # interés previo (no observado)
logit_base = -2.2 + 0.9 * interes
u = rng_pob.random(M)
y_sin = (u < 1 / (1 + np.exp(-logit_base))).astype(int)            # compra SIN el botón nuevo
y_con = (u < 1 / (1 + np.exp(-(logit_base + 0.25)))).astype(int)   # compra CON el botón nuevo
efecto_verdadero = (y_con - y_sin).mean() * 100

# A) Asignación aleatoria (moneda)
trat_alea = rng_pob.binomial(1, 0.5, M)
y_alea = np.where(trat_alea == 1, y_con, y_sin)
dif_alea = (y_alea[trat_alea == 1].mean() - y_alea[trat_alea == 0].mean()) * 100

# B) Autoselección: los más interesados usan más el botón
p_uso = 1 / (1 + np.exp(-1.2 * interes))
trat_auto = rng_pob.binomial(1, p_uso)
y_auto = np.where(trat_auto == 1, y_con, y_sin)
dif_auto = (y_auto[trat_auto == 1].mean() - y_auto[trat_auto == 0].mean()) * 100

print(f"Efecto verdadero (puntos porcentuales): {efecto_verdadero:.2f}")
print(f"Estimado con aleatorización:      {dif_alea:.2f}")
print(f"Estimado con autoselección:       {dif_auto:.2f}")
print(f"Interés previo medio, autoselección (tratados / no tratados): {interes[trat_auto == 1].mean():.2f} / {interes[trat_auto == 0].mean():.2f}")
print(f"Interés previo medio, aleatorización (tratados / no tratados): {interes[trat_alea == 1].mean():.2f} / {interes[trat_alea == 0].mean():.2f}")

fig, ax = plt.subplots()
ax.bar(["Verdadero", "Aleatorio", "Autoselección"], [efecto_verdadero, dif_alea, dif_auto], color=["gray", "tab:green", "tab:red"])
ax.set_ylabel("Efecto estimado (puntos porcentuales)")
ax.set_title("Mismo botón, dos formas de repartirlo")
plt.show()

**¿Qué observas?** Con la moneda el grupo tratado y el de control tienen el mismo interés previo (diferencia ≈ 0) y la estimación cae muy cerca del efecto verdadero. Con autoselección los tratados eran ya más propensos a comprar y el "efecto" sale unas cuatro veces mayor: es **correlación**, no causa (como vimos en la clase 10). La aleatorización equilibra, **en promedio**, todo lo que no medimos.

---
## 2. Diseñar antes de empezar: el tamaño de muestra

Decidimos **antes** de ver datos: métrica primaria (conversión), unidad de aleatorización (visitante), α = 0,05, potencia 80 % y **efecto mínimo relevante** (MDE): pasar de 10 % a 12 % (+2 puntos). La fórmula, para dos proporciones y grupos iguales:

`n = (z(1−α/2) + z(potencia))² · [p₁(1−p₁) + p₂(1−p₂)] / (p₂ − p₁)²`

Primero a mano y después con `statsmodels` (`proportion_effectsize` da el tamaño de efecto *h* de Cohen).

In [ ]:
p_base, p_obj = 0.10, 0.12
za = stats.norm.ppf(1 - 0.05 / 2)
zb = stats.norm.ppf(0.80)
suma_var = p_base * (1 - p_base) + p_obj * (1 - p_obj)
n_mano = (za + zb) ** 2 * suma_var / (p_obj - p_base) ** 2
print(f"z(0,975) = {za:.3f}   z(0,80) = {zb:.3f}   (za+zb)^2 = {(za + zb) ** 2:.3f}   suma de varianzas = {suma_var:.4f}")
print(f"n por grupo a mano: {n_mano:.1f}")

h = proportion_effectsize(p_obj, p_base)
n_sm = NormalIndPower().solve_power(effect_size=h, alpha=0.05, power=0.80, ratio=1)
print(f"h = {h:.4f}   n por grupo con statsmodels: {n_sm:.1f}")
pot_2000 = NormalIndPower().solve_power(effect_size=h, nobs1=2000, alpha=0.05, ratio=1)
print(f"Potencia si solo tuviéramos 2 000 por grupo: {pot_2000 * 100:.1f} %")

La fórmula a mano y `statsmodels` difieren en unas pocas unidades (usan aproximaciones ligeramente distintas): redondeamos hacia arriba a **3 840 por grupo**. Con la mitad de visitantes la potencia cae a ~53 %: casi una moneda al aire.

### ¿Cómo cambia n?

**Antes de ejecutar, predice:** si el efecto a detectar se **duplica** (de +2 a +4 puntos), ¿por cuánto se multiplica n? ¿Y si se reduce a la mitad?

In [ ]:
filas = []
for base, mde in [(0.10, 0.01), (0.10, 0.02), (0.10, 0.03), (0.10, 0.04)]:
    filas.append({"escenario": f"{base:.0%} → {base + mde:.0%}", "MDE (pp)": mde * 100, "n": round(n_por_grupo(base, base + mde))})
tab_mde = pd.DataFrame(filas)
tab_mde["n relativo a +2 pp"] = (tab_mde["n"] / tab_mde.loc[1, "n"]).round(2)
display(tab_mde)

variantes = pd.DataFrame({
    "caso": ["referencia (10 % → 12 %, α 5 %, potencia 80 %)", "potencia 90 %", "α = 1 %",
             "tasa base 5 % → 6 % (+20 % relativo)", "tasa base 20 % → 24 % (+20 % relativo)"],
    "n": [round(n_por_grupo(0.10, 0.12)), round(n_por_grupo(0.10, 0.12, potencia=0.90)), round(n_por_grupo(0.10, 0.12, alfa=0.01)),
          round(n_por_grupo(0.05, 0.06)), round(n_por_grupo(0.20, 0.24))]})
display(variantes)

mdes = np.linspace(0.01, 0.06, 50)
fig, ax = plt.subplots()
ax.plot(mdes * 100, [n_por_grupo(0.10, 0.10 + m) for m in mdes])
ax.set_yscale("log")
ax.set_xlabel("Efecto mínimo a detectar (puntos porcentuales)")
ax.set_ylabel("n por grupo (escala log)")
ax.set_title("Tasa base 10 %: el efecto pequeño es carísimo")
plt.show()

**¿Qué observas?** Duplicar el efecto reduce n a **aproximadamente una cuarta parte** (n va con 1/efecto²: el cociente sale 0,27 y no 0,25 exactamente porque las varianzas también cambian). Pasar de +2 a +1 punto lo cuadruplica. Exigir más potencia o menos α también encarece; y a igual mejora *relativa*, una tasa base baja necesita muchos más visitantes.

### Experimento guiado 2: comprobar la potencia simulando

Repetimos 4 000 veces el experimento con efecto **real** de +2 puntos y contamos cuántas veces el test sale significativo con n = 3 840 y con n = 2 000. **Antes de ejecutar, predice** los porcentajes.

In [ ]:
rng_pot = np.random.default_rng(5)
for n in (3840, 2000):
    xa = rng_pot.binomial(n, 0.10, 4000)
    xb = rng_pot.binomial(n, 0.12, 4000)
    z = z_dos_proporciones(xa, n, xb, n)
    print(f"n = {n}:  experimentos significativos (p < 0,05): {(np.abs(z) > 1.96).mean() * 100:.1f} %")

**¿Qué observas?** Con n = 3 840 el test detecta el efecto en torno al 80 % de las veces (eso es la potencia); con 2 000, en torno a la mitad: en el otro 50 % existe un efecto real y el test dice "no significativo". Es el error de tipo II de la clase 7, ahora con número.

### Duración

Con 800 visitantes al día en total (*ejemplo ilustrativo*) hacen falta 2 × 3 840 = 7 680 visitantes: 9,6 días. Se redondea a **semanas completas** (14 días), porque el comportamiento cambia entre semana y fin de semana y no conviene cortar el experimento a mitad de un ciclo.

---
## 3. Analizar un test A/B de conversión

Ejecutamos el experimento planificado: 3 840 visitantes por grupo, con tasas **verdaderas** 10 % (A, control) y 12 % (B). Los datos son sintéticos, en formato de una fila por visitante.

In [ ]:
n_grupo = 3840
rng_ab = np.random.default_rng(4)
conv_A = rng_ab.random(n_grupo) < 0.10
conv_B = rng_ab.random(n_grupo) < 0.12
datos_ab = pd.DataFrame({"grupo": ["A"] * n_grupo + ["B"] * n_grupo,
                         "convierte": np.concatenate([conv_A, conv_B]).astype(int)})
resumen = datos_ab.groupby("grupo")["convierte"].agg(["sum", "count", "mean"])
display(resumen)

a, b = int(resumen.loc["A", "sum"]), int(resumen.loc["B", "sum"])
pa, pb = a / n_grupo, b / n_grupo
z, p_valor = proportions_ztest([b, a], [n_grupo, n_grupo])
print(f"Diferencia B − A: {(pb - pa) * 100:.2f} pp   z = {z:.3f}   p-value = {p_valor:.4f}")

# IC 95 % de la diferencia: a mano (normal) y con statsmodels (Newcombe, basado en Wilson)
ee = np.sqrt(pa * (1 - pa) / n_grupo + pb * (1 - pb) / n_grupo)
print(f"IC a mano: [{((pb - pa) - 1.96 * ee) * 100:.2f}, {((pb - pa) + 1.96 * ee) * 100:.2f}] pp")
lo, hi = confint_proportions_2indep(b, n_grupo, a, n_grupo, method="newcomb")
print(f"IC Newcombe: [{lo * 100:.2f}, {hi * 100:.2f}] pp")

# Lift (mejora relativa): cociente pB/pA, con su IC
lo_r, hi_r = confint_proportions_2indep(b, n_grupo, a, n_grupo, compare="ratio")
print(f"Lift = {(pb / pa - 1) * 100:.1f} %   IC 95 %: [{(lo_r - 1) * 100:.1f} %, {(hi_r - 1) * 100:.1f} %]")

# Traducido a negocio: por cada 100 000 visitantes (ejemplo ilustrativo)
print(f"Conversiones extra por 100 000 visitantes: {(pb - pa) * 1e5:.0f}   (IC: {((pb - pa) - 1.96 * ee) * 1e5:.0f} a {((pb - pa) + 1.96 * ee) * 1e5:.0f})")

**¿Qué observas?** El resultado es significativo (p ≈ 0,02), el IC de la diferencia no incluye el 0 y el lift es de ~16 %. Pero el efecto observado (+1,7 pp) es algo menor que el MDE planificado (+2 pp) y el IC es ancho: va de +0,3 a +3,1 puntos. **Significativo no significa importante:** el valor más bajo del IC podría no compensar el coste del cambio. Y si el test no hubiera salido significativo, eso **no** habría demostrado que B sea igual que A: solo que no tenemos evidencia suficiente.

---
## 4. El peeking: mirar antes de tiempo

Tentación clásica: revisar el panel cada día y **parar en cuanto p < 0,05**. Cada mirada es una nueva oportunidad de falso positivo, y las oportunidades se acumulan. Lo comprobamos con experimentos **A/A**: ambos grupos con la **misma** tasa real (10 %), de modo que cada "victoria" es un falso positivo por definición.

### Experimento guiado 3: falsos positivos con controles repetidos

Simulamos 4 000 experimentos A/A de 3 840 visitantes por grupo. En cada uno miramos 1, 2, 5, 10 o 20 veces (a intervalos regulares) y lo declaramos "ganador" si en **cualquier** control |z| > 1,96. **Antes de ejecutar, predice:** con un único control esperamos 5 %; ¿y con 10 o 20?

In [ ]:
rng_peek = np.random.default_rng(7)
controles_lista = [1, 2, 5, 10, 20]
tasas = [falsos_positivos(rng_peek, 4000, L) for L in controles_lista]
tab_peek = pd.DataFrame({"controles": controles_lista, "% de falsos positivos": np.round(tasas, 1)})
display(tab_peek)

fig, ax = plt.subplots()
ax.plot(controles_lista, tasas, "o-", color="tab:red")
ax.axhline(5, color="gray", linestyle="--", label="nivel nominal 5 %")
ax.set_xlabel("Número de controles")
ax.set_ylabel("Falsos positivos reales (%)")
ax.set_title("Sin efecto real, mirar y parar «gana» cada vez más")
ax.legend()
plt.show()

**¿Qué observas?** Con un control el 5 % nominal se cumple; con 5 controles ronda el 13 %, con 10 casi el 19 % y con 20 alrededor del 25 %: **uno de cada cuatro** experimentos sin efecto acaba "ganando". Veamos ahora cómo se ve una de esas trayectorias, y un remedio.

In [ ]:
z_todos = z_controles(np.random.default_rng(3), 300, 20)
cruza = (np.abs(z_todos) > 1.96).any(axis=1) & (np.abs(z_todos[:, -1]) < 1.96)
idx = int(np.argmax(cruza))              # primer experimento que cruza el umbral pero termina "no significativo"
fig, ax = plt.subplots()
ax.plot(range(1, 21), z_todos[idx], "o-")
ax.axhline(1.96, color="red", linestyle="--"); ax.axhline(-1.96, color="red", linestyle="--")
ax.set_xlabel("Control (cada uno con más datos)")
ax.set_ylabel("z")
ax.set_title("Un experimento A/A: cruza el umbral y termina sin efecto")
plt.show()
print("¿Cruza en algún control?", bool(cruza[idx]), "  |  ¿Significativo al final?", bool(abs(z_todos[idx, -1]) > 1.96))

# Remedio conceptual: umbral más exigente en cada control (Pocock, 5 controles: z ≈ 2,413)
print("5 controles con |z| > 1,96:", round(falsos_positivos(np.random.default_rng(8), 4000, 5), 1), "%")
print("5 controles con |z| > 2,413:", round(falsos_positivos(np.random.default_rng(8), 4000, 5, umbral=2.413), 1), "%")

**¿Qué observas?** La trayectoria cruza el umbral en algún momento y, con todos los datos, se queda dentro: un "ganador" habría sido un falso positivo. Subir el umbral en cada control (idea de los **test secuenciales** y del *alpha spending*) devuelve la tasa a ~5 %, pero es un método con reglas propias que debe definirse **antes** del experimento. La solución sencilla: fijar n de antemano y analizar una sola vez al final.

---
## 5. Controles de salud del experimento

Antes de leer el resultado hay que comprobar que el experimento **se ejecutó bien**.

### SRM: sample ratio mismatch

Si diseñamos un reparto 50/50, los tamaños observados deben parecerse a 50/50. Se comprueba con un **chi-cuadrado de bondad de ajuste** (clase 9). Un p-value muy pequeño (umbral habitual: 0,001) indica un problema de asignación o de registro: el resultado del test no es fiable aunque salga "significativo".

In [ ]:
def control_srm(obs, prop_esperada=None):
    obs = np.array(obs)
    esperado = obs.sum() * (np.array(prop_esperada) if prop_esperada is not None else np.full(len(obs), 1 / len(obs)))
    chi2, p = stats.chisquare(obs, f_exp=esperado)
    return chi2, p

for nombre, obs in [("Experimento X", [20400, 19600]), ("Experimento Y", [10050, 9950])]:
    chi2, p = control_srm(obs)
    print(f"{nombre}: {obs}  chi2 = {chi2:.2f}  p-value = {p:.5f}  →  {'ALERTA SRM' if p < 0.001 else 'sin señal de SRM'}")
# A mano para X: (400²/20000)·2
print("Comprobación a mano de X:", 2 * 400 ** 2 / 20000)

**¿Qué observas?** X tiene una diferencia de solo 2 % entre grupos, pero con 40 000 visitantes es imposible por azar (p ≈ 0,00006): algo falla (un *bug* en la asignación, bots, un redireccionamiento lento en B…). Hay que **arreglar la causa**, no «corregir» estadísticamente. En cambio Y tiene una diferencia del 1 % que el azar explica bien (p ≈ 0,48).

### Test A/A

Antes de confiar en la plataforma de experimentos se ejecutan tests **A/A** (dos grupos idénticos): los p-values deben ser uniformes y ~5 % debe salir por debajo de 0,05.

In [ ]:
rng_aa = np.random.default_rng(11)
xa = rng_aa.binomial(3840, 0.10, 5000)
xb = rng_aa.binomial(3840, 0.10, 5000)
p_aa = 2 * stats.norm.sf(np.abs(z_dos_proporciones(xa, 3840, xb, 3840)))
print(f"Tests A/A con p < 0,05: {(p_aa < 0.05).mean() * 100:.1f} %   (esperado ≈ 5 %)")
fig, ax = plt.subplots()
ax.hist(p_aa, bins=10, range=(0, 1), edgecolor="white")
ax.set_xlabel("p-value"); ax.set_ylabel("Número de tests A/A")
ax.set_title("Sin efecto real, los p-values son uniformes")
plt.show()

**¿Qué observas?** Los p-values se reparten de forma casi uniforme y ~5 % cae bajo 0,05: la tubería de análisis está sana. Si vieras un exceso de valores pequeños, habría un problema (medición, unidad de análisis…). Un tercer control, que solo mencionamos, es el **efecto novedad**: un cambio llamativo puede rendir más al principio solo por ser nuevo; mirar si el efecto se mantiene por semanas ayuda a detectarlo.

---
## 6. La paradoja de Simpson

Una tabla pequeña que se verifica a mano: dos segmentos de dispositivo, con el grupo B ganando en **cada** segmento y perdiendo en el **total**. (*Ejemplo ilustrativo*, números inventados para la demostración.)

In [ ]:
tabla = pd.DataFrame({"segmento": ["desktop", "desktop", "mobile", "mobile"],
                      "grupo": ["A", "B", "A", "B"],
                      "usuarios": [300, 100, 100, 300],
                      "conversiones": [90, 33, 5, 18]})
tabla["tasa %"] = (tabla["conversiones"] / tabla["usuarios"] * 100).round(2)
display(tabla)

total = tabla.groupby("grupo")[["usuarios", "conversiones"]].sum()
total["tasa %"] = (total["conversiones"] / total["usuarios"] * 100).round(2)
display(total)

# Comparación estandarizada: mismo peso de cada segmento para A y para B (total de usuarios del segmento)
pesos = tabla.groupby("segmento")["usuarios"].sum() / tabla["usuarios"].sum()
tasa = tabla.pivot(index="segmento", columns="grupo", values="conversiones") / tabla.pivot(index="segmento", columns="grupo", values="usuarios")
estand = (tasa.mul(pesos, axis=0)).sum() * 100
print("Tasas estandarizadas (%):", estand.round(2).to_dict(), "  diferencia B − A:", round(estand["B"] - estand["A"], 2), "pp")

print("Composición de usuarios por grupo (% desktop):",
      {g: round(tabla[(tabla.grupo == g) & (tabla.segmento == "desktop")]["usuarios"].iloc[0] / 400 * 100) for g in ("A", "B")})

**¿Qué observas?** B convierte más en desktop (33 % frente a 30 %) y en mobile (6 % frente a 5 %), pero pierde en el total (12,75 % frente a 23,75 %). La causa: el 75 % de los usuarios de A son de desktop, que convierte mucho, y solo el 25 % de los de B. **Los grupos no son comparables**: en un experimento bien aleatorizado esa diferencia de composición no debería aparecer, y si aparece es una señal de fallo (como un SRM por segmento). Al comparar con el mismo peso para cada segmento, B gana por +2 pp. Para evitarlo: aleatorizar bien, comprobar el equilibrio por segmento, estratificar y analizar los segmentos con cautela.

---
## 7. Segmentos y métricas múltiples

"No ha ganado en general, pero mira qué bien va en el segmento 7." Con 10 segmentos (o 10 métricas) y α = 0,05, la probabilidad de al menos un falso positivo es 1 − 0,95¹⁰ ≈ 40 %. Es el problema de las comparaciones múltiples de la clase 9.

### Experimento guiado 4: 10 segmentos sin efecto

Simulamos 2 000 experimentos A/A con 10 segmentos de 1 000 usuarios por grupo. Contamos cuántas veces algún segmento sale "significativo" con p sin corregir y con la corrección de Holm. **Antes de ejecutar, predice** ambos porcentajes.

In [ ]:
rng_seg = np.random.default_rng(13)
n_sim, n_seg, n_u = 2000, 10, 1000
xa = rng_seg.binomial(n_u, 0.10, (n_sim, n_seg))
xb = rng_seg.binomial(n_u, 0.10, (n_sim, n_seg))
p_seg = 2 * stats.norm.sf(np.abs(z_dos_proporciones(xa, n_u, xb, n_u)))
alguno_sin_corregir = (p_seg < 0.05).any(axis=1).mean() * 100
alguno_holm = np.mean([multipletests(fila, alpha=0.05, method="holm")[0].any() for fila in p_seg]) * 100
print(f"Teórico 1 − 0,95^10: {(1 - 0.95 ** 10) * 100:.1f} %")
print(f"Algún segmento significativo, sin corregir: {alguno_sin_corregir:.1f} %")
print(f"Algún segmento significativo, con Holm: {alguno_holm:.1f} %")

**¿Qué observas?** Sin corregir, en torno a 4 de cada 10 experimentos **sin ningún efecto** presentan algún segmento "ganador"; con Holm, ~5 %. Regla práctica: define **una** métrica primaria y una lista corta de métricas de control (*guardrails*) antes de empezar; trata los análisis por segmento como exploratorios, o corrígelos.

---
## 8. Resumen

- **Aleatorizar** permite atribuir la diferencia al tratamiento, en promedio, si el experimento se ejecuta bien.
- **Diseña antes:** métrica primaria, MDE, α, potencia y n (`NormalIndPower`, `proportion_effectsize`). Duplicar el efecto reduce n a ~1/4.
- **Analiza una vez**, con test de dos proporciones, IC de la diferencia y lift; significativo ≠ importante; no significativo ≠ "sin efecto".
- **Peeking**: con 20 controles el falso positivo real ronda el 25 %.
- **Controles de salud:** SRM (chi-cuadrado), A/A, efecto novedad.
- **Simpson** y **segmentos/métricas múltiples**: compara grupos comparables y corrige las comparaciones.

Continúa con la clase 14: Proyecto integrador. Ahora pasa al notebook de práctica.